# 03. 유명한 포트폴리오 알고리즘 돌려보기

**대응 강의:** [3강. 유명 알고리즘으로 갈아끼우기](../Lecture/03.md) · **이전 실습:** [01_basic.ipynb](01_basic.ipynb)

1강에선 비중을 무작위로 2만 번 뽑아 가장 좋은 걸 골랐죠. 솔직히 좀 '맨손 탐색'이었어요.
이번엔 같은 문제를 **세상이 검증한 유명 알고리즘 3개**로 풀고, 어느 게 좋은지 **한 표로 비교**해볼 거예요.

- 자산도 2강에서 배운 대로 **성격이 다른 ETF 5개**로 제대로 분산했어요.
- 라이브러리(`PyPortfolioOpt`)를 쓰니 알고리즘은 **목표만 바꿔 끼우면** 돼요.

> ▶️ 위에서부터 차례로 실행하세요. 1번 셀(설치)과 데이터 받는 셀은 살짝 시간이 걸려요.

In [ ]:
# Colab엔 PyPortfolioOpt가 기본으로 안 깔려 있어요. 한 번만 설치하면 돼요.
!pip install -q PyPortfolioOpt yfinance

## 1. 데이터 — 성격 다른 자산 5종

2강의 교훈을 적용했어요. 빅테크만 모으는 대신, **서로 다르게 움직이는** 자산군을 골랐어요:

| 티커 | 자산 | 성격 |
|---|---|---|
| `SPY` | 미국 주식 | 성장 엔진 |
| `AGG` | 미국 채권 | 방어·안정 |
| `GLD` | 금 | 위기 때 헤지 |
| `VNQ` | 부동산(REITs) | 인플레 대응 |
| `EFA` | 선진국(미국 외) 주식 | 지역 분산 |

이렇게 골라야 공분산이 낮아지고, 최적화가 위험을 진짜로 줄일 수 있어요.

In [ ]:
import yfinance as yf
import numpy as np
import pandas as pd
from pypfopt import EfficientFrontier, risk_models, expected_returns, HRPOpt

# 성격 다른 자산군 ETF 5종을 받아와요 (수정 종가 기준)
prices = yf.download(['SPY', 'AGG', 'GLD', 'VNQ', 'EFA'],
                     start='2018-01-01', auto_adjust=True, progress=False)['Close']

print('데이터 모양:', prices.shape)
prices.tail(3)

## 2. 입력 추정 — 알고리즘에 넣을 '재료'

모든 최적화 알고리즘은 두 가지 재료를 먹어요: **기대수익(`mu`)** 과 **위험(`S`, 공분산)**.

- `mean_historical_return`: 과거 평균 수익률을 연율화해서 기대수익으로 써요.
- `CovarianceShrinkage(...).ledoit_wolf()`: 그냥 표본 공분산은 데이터가 적으면 들쭉날쭉해요.
  **수축(shrinkage)** 은 극단적인 값을 살짝 눌러서 **더 안정적인 공분산**을 만들어줘요.

In [ ]:
mu = expected_returns.mean_historical_return(prices)        # 기대수익 (연율화)
S = risk_models.CovarianceShrinkage(prices).ledoit_wolf()   # 공분산 안정화(수축)

print('기대수익(mu, 연율화):')
print(mu.round(3))

## 3. 알고리즘 3개 — 목표만 바꿔 끼우기

같은 재료(`mu`, `S`)를 주고 **'무엇을 최고로 칠지'(목표)만** 바꿔봐요. 그 목표가 곧 알고리즘이에요.

| 알고리즘 | 목표 | 한 줄 성격 |
|---|---|---|
| **최대 샤프** | 위험 대비 수익 최대 | "효율이 제일 좋은 조합" |
| **최소분산** | 변동성 최소 | "수익보다 안 흔들리는 게 우선" |
| **HRP** | 자산을 끼리끼리 묶어 위험 분배 | "머신러닝(군집)으로 분산", 추정 오차에 강함 |

`weight_bounds=(0, 0.35)`는 **한 자산에 35% 넘게 몰빵 금지**라는 제약이에요.

In [ ]:
# 3-A. 최대 샤프 — 위험 대비 수익이 가장 좋은 조합
ef = EfficientFrontier(mu, S, weight_bounds=(0, 0.35))
ef.max_sharpe()
print('최대 샤프 비중:', ef.clean_weights())
ef.portfolio_performance(verbose=True)

In [ ]:
# 3-B. 최소분산 — 목표만 'min_volatility'로 교체 (새 객체로 다시 시작해야 해요)
ef2 = EfficientFrontier(mu, S, weight_bounds=(0, 0.35))
ef2.min_volatility()
print('최소분산 비중:', ef2.clean_weights())
ef2.portfolio_performance(verbose=True)

In [ ]:
# 3-C. HRP — 자산을 군집으로 묶어 위험을 나눠 담는 머신러닝 기반 방법
returns = prices.pct_change(fill_method=None).dropna()
hrp = HRPOpt(returns)
hrp_weights = hrp.optimize()
print('HRP 비중:', hrp_weights)
hrp.portfolio_performance(verbose=True)

## 4. 분석 — 한 표로 비교 (벤치마크 포함)

비중만 봐선 누가 나은지 몰라요. **같은 잣대로 점수**를 매겨 비교해요. 특히 두 가지가 중요해요:

- **최대낙폭(MDD)**: 고점 대비 가장 크게 빠졌던 낙폭. "최악일 때 얼마나 아팠나"예요.
- **균등비중(1/N)**: 그냥 5개에 똑같이 20%씩. **"머리 안 쓴 기본값"** 벤치마크예요.
  똑똑한 알고리즘이라면 적어도 이건 이겨야겠죠?

In [ ]:
def evaluate(weights, returns, rf=0.02):
    w = np.array([weights[t] for t in returns.columns])
    port = returns @ w                                  # 포트폴리오 일간 수익
    ann_ret = port.mean() * 252                         # 연율화 수익률
    ann_vol = port.std() * np.sqrt(252)                 # 연율화 변동성
    sharpe = (ann_ret - rf) / ann_vol
    cum = (1 + port).cumprod()                          # 누적 자산 곡선
    mdd = ((cum - cum.cummax()) / cum.cummax()).min()   # 최대 낙폭(MDD)
    return {'수익률': ann_ret, '변동성': ann_vol, '샤프': sharpe, '최대낙폭': mdd}

In [ ]:
# 알고리즘 3개 + 균등비중 벤치마크를 한 표로 모아요
n = len(prices.columns)
equal_w = {t: 1 / n for t in prices.columns}            # 1/N 벤치마크

rows = {
    '최대샤프': evaluate(ef.clean_weights(), returns),
    '최소분산': evaluate(ef2.clean_weights(), returns),
    'HRP':     evaluate(hrp_weights, returns),
    '균등비중': evaluate(equal_w, returns),
}

table = pd.DataFrame(rows).T
# 보기 좋게 %와 소수점 정리
fmt = table.copy()
for c in ['수익률', '변동성', '최대낙폭']:
    fmt[c] = (table[c] * 100).round(1).astype(str) + '%'
fmt['샤프'] = table['샤프'].round(2)
fmt

## 정리

- 최적화 알고리즘은 결국 **'무엇을 최고로 칠지'(목표)를 바꾸는 것**이에요. 재료(`mu`, `S`)는 그대로고요.
- **최대샤프**는 효율, **최소분산**은 안정, **HRP**는 추정 오차에 강한 분산 — 성격이 서로 달라요.
- 무조건 좋은 알고리즘은 없어요. **균등비중(1/N)** 과 비교해서 "정말 더 나은지"를 늘 확인하세요.
- 이 비교가 바로 [2강](../Lecture/02.md) 플로우의 **⑤ 백테스트·검증** 단계예요.

> 💡 의외로 실전에선 단순한 **1/N**이 똑똑한 알고리즘을 이기는 경우가 꽤 많아요.
> 그만큼 '기대수익(mu) 추정'이 어렵다는 뜻이에요.

## 🎯 직접 해보기

1. `weight_bounds`를 `(0, 1)`로 풀면(몰빵 허용) 최대샤프 비중이 어떻게 쏠리나요?
2. 시작일을 `2018` 대신 `2007`(금융위기 포함)로 바꾸면 최소분산의 최대낙폭이 얼마나 달라지나요?
3. 표에서 **샤프가 가장 높은 전략**과 **최대낙폭이 가장 작은 전략**이 같은가요? 다르다면 왜일까요?